**Fraud Detection in E-Commerce Transactions**

Goal:
Detect fraudulent transactions using unsupervised and ensemble machine learning, simulate near-real-time processing, and quantify business impact.

# Dataset Exploration & Understanding
Dataset

Use a public fraud dataset (recommended):
<ul>
    <li>IEEE-CIS Fraud Detection</li>
    <li>PaySim (Mobile Money Fraud Simulation)</li>
<ul>
<br>
For simplicity and reproducibility, we use PaySim.

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("ealaxi/paysim1")

# print("Path to dataset files:", path)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("PS_20174392719_1491204439457_log.csv")
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [5]:
df.shape

(6362620, 11)

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [3]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [ ]:
df.sample(2000)

In [2]:
df.isnull().sum()
df['isFraud'].value_counts(normalize=True)

isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64

# Feature Engineering

In [3]:
# Amount-Based Features
df['log_amount'] = np.log1p(df['amount'])

In [4]:
# Transaction Frequency (User Behavior)
df['tx_count_orig'] = df.groupby('nameOrig')['amount'].transform('count')
df['tx_sum_orig'] = df.groupby('nameOrig')['amount'].transform('sum')

In [5]:
# Device / Behavioral Proxy Features
df['amount_to_avg_ratio'] = df['amount'] / (df['tx_sum_orig'] / df['tx_count_orig'])


In [6]:
features = [
    'amount', 'log_amount',
    'tx_count_orig',
    'tx_sum_orig',
    'amount_to_avg_ratio'
]
X = df[features]
y = df['isFraud']

In [9]:
X_filled = X.fillna(X.mean())

 # Model Training (Isolation Forest + Ensemble)

In [10]:
# Isolation Forest (Primary Fraud Detector)

from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42
)

df['anomaly_score'] = iso.fit_predict(X_filled)
df['iso_fraud'] = (df['anomaly_score'] == -1).astype(int)

In [ ]:
# Optional Ensemble (Isolation Forest + Random Forest)
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced')
rf.fit(X, y)

df['rf_fraud'] = rf.predict(X)

# Ensemble decision
df['ensemble_fraud'] = (
    (df['iso_fraud'] == 1) | (df['rf_fraud'] == 1)
).astype(int)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y, df['ensemble_fraud']))

# Stream Processing Simulation

Simulate real-time transaction ingestion.

In [ ]:
import time

def process_transaction(tx):
    score = iso.predict([tx])[0]
    return 1 if score == -1 else 0

for _, row in X.head(100).iterrows():
    fraud = process_transaction(row)
    print("Fraud Detected" if fraud else "Transaction OK")
    time.sleep(0.1)

# KPI & Business Impact Report

In [ ]:
# Fraud Detection Rate
fraud_detected = df[df['ensemble_fraud'] == 1]

# Fraud Savings Impact
total_fraud_loss = df[df['isFraud']==1]['amount'].sum()
prevented_loss = fraud_detected['amount'].sum()

print("Total Fraud Loss:", total_fraud_loss)
print("Prevented Loss:", prevented_loss)